In [ ]:
!pip install xgboost shap scikit-learn pandas numpy matplotlib pyarrow
!pip install pytorch-forecasting lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 64.8 MB/s eta 0:00:00


Note: I used the dataset.parquet generated.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CONFIG = {
    "dataset": "/content/dataset.parquet",
    "crop":    "corn",
    "out_dir": "./results",
}

# ── IMPORTS ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr

ANNUALISATION = np.sqrt(52)
VOL_WINDOW    = 12

# ── LOAD ──────────────────────────────────────────────────────────────────────
df = pd.read_parquet(CONFIG["dataset"])
df.index = pd.to_datetime(df.index)
df = df.sort_index()
crop = CONFIG["crop"]
print(f"Loaded: {df.shape}  ({df.index.min().date()} → {df.index.max().date()})")
print(f"Columns: {df.columns.tolist()}")

# ── FEATURE ENGINEERING (no external data needed) ────────────────────────────
def engineer_features(df, crop):
    df = df.copy()
    ret = f"{crop}_ret_1w"

    # ── Extra momentum lags ────────────────────────────────────────────────
    for lag in [2, 3, 4, 8]:
        df[f"{crop}_ret_{lag}w_lag"] = df[ret].shift(lag)

    # ── Realised vol at multiple windows ──────────────────────────────────
    for w in [4, 8, 12, 26]:
        df[f"{crop}_rv_{w}w"]        = df[ret].rolling(w).std() * ANNUALISATION
        rv_mean = df[f"{crop}_rv_{w}w"].rolling(52).mean()
        rv_std  = df[f"{crop}_rv_{w}w"].rolling(52).std() + 1e-9
        df[f"{crop}_rv_z_{w}w"]      = (df[f"{crop}_rv_{w}w"] - rv_mean) / rv_std

    # ── Vol regime flag: 1 = high vol, 0 = low vol ────────────────────────
    rv12 = df[f"{crop}_rv_12w"]
    df[f"{crop}_high_vol_regime"] = (rv12 > rv12.rolling(52).quantile(0.7)).astype(int)

    # ── Trend strength (ADX proxy) ─────────────────────────────────────────
    for w in [4, 12]:
        up   = df[ret].clip(lower=0).rolling(w).mean()
        down = (-df[ret]).clip(lower=0).rolling(w).mean() + 1e-9
        df[f"{crop}_rsi_proxy_{w}w"] = up / (up + down)

    # ── Cross-asset features ───────────────────────────────────────────────
    others = [c for c in ["corn", "wheat", "soybean"] if c != crop]
    for other in others:
        other_ret = f"{other}_ret_1w"
        if other_ret in df.columns:
            # Relative return vs peer
            df[f"{crop}_vs_{other}_ret"] = df[ret] - df[other_ret]
            # Correlation regime: 4w rolling corr with peer
            df[f"{crop}_{other}_corr_8w"] = (
                df[ret].rolling(8).corr(df[other_ret]))
            # Does peer momentum predict our next move?
            for lag in [1, 2]:
                df[f"{other}_ret_lag{lag}"] = df[other_ret].shift(lag)

    # ── Grain complex momentum ─────────────────────────────────────────────
    grain_cols = [f"{c}_ret_1w" for c in ["corn", "wheat", "soybean"]
                  if f"{c}_ret_1w" in df.columns]
    df["grain_complex_ret"] = df[grain_cols].mean(axis=1)
    df[f"{crop}_rel_strength"] = df[ret] - df["grain_complex_ret"]

    # ── Mean reversion signal ──────────────────────────────────────────────
    for w in [4, 8]:
        roll_mean = df[ret].rolling(w).mean()
        roll_std  = df[ret].rolling(w).std() + 1e-9
        df[f"{crop}_zscore_{w}w"] = (df[ret] - roll_mean) / roll_std

    # ── Seasonal interactions ──────────────────────────────────────────────
    # Multiply seasonal flags with vol — vol behaves differently by season
    for flag in ["is_planting", "is_growing", "is_harvest"]:
        if flag in df.columns:
            df[f"{crop}_rv_x_{flag}"] = df[f"{crop}_rv_12w"] * df[flag]

    return df

df = engineer_features(df, crop)
print(f"After feature engineering: {df.shape}")

# ── FEATURES & TARGET ─────────────────────────────────────────────────────────
target_col = f"{crop}_target_4w"
exclude     = [c for c in df.columns if
               "_target_" in c or "_direction_" in c
               or c in ["corn", "wheat", "soybean"]]
feature_cols = [c for c in df.columns
                if c not in exclude
                and pd.api.types.is_numeric_dtype(df[c])]
print(f"Features: {len(feature_cols)}")

# ── SPLITS ────────────────────────────────────────────────────────────────────
TRAIN_END = "2018-12-31"
VAL_END   = "2021-12-31"
train = df.loc[:TRAIN_END].dropna(subset=[target_col])
val   = df.loc[TRAIN_END:VAL_END].iloc[1:].dropna(subset=[target_col])
test  = df.loc[VAL_END:].iloc[1:].dropna(subset=[target_col])
print(f"Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

X_tr = train[feature_cols].fillna(0).values;  y_tr = train[target_col].values
X_v  = val[feature_cols].fillna(0).values;    y_v  = val[target_col].values
X_te = test[feature_cols].fillna(0).values;   y_te = test[target_col].values

# ── SHARPE OBJECTIVE ──────────────────────────────────────────────────────────
def sharpe_obj(y_pred, dtrain):
    y_true = dtrain.get_label()
    eps = 1e-6
    r   = y_pred * y_true
    mu  = r.mean();  sigma = r.std() + eps
    grad = -(1/len(y_pred)) * y_true / sigma \
           + (mu/(len(y_pred)*sigma**2)) * ((r-mu)/sigma) * y_true
    hess = np.abs(grad) + eps
    return grad, hess

def sharpe_metric(y_pred, dtrain):
    y_true = dtrain.get_label()
    r      = np.sign(y_pred) * y_true
    sharpe = r.mean() / (r.std() + 1e-9) * ANNUALISATION
    return "neg_sharpe", -float(sharpe)

# ── TRAIN: PHASE 1 warmup (RMSE) ─────────────────────────────────────────────
print("\nPhase 1: RMSE warmup...")
params = dict(
    objective    = "reg:squarederror",
    max_depth    = 3,
    learning_rate= 0.02,
    subsample    = 0.7,
    colsample_bytree = 0.6,
    min_child_weight = 8,
    reg_alpha    = 0.5,
    reg_lambda   = 2.0,
    seed=42, nthread=-1,
)
dtrain = xgb.DMatrix(X_tr, label=y_tr)
dval   = xgb.DMatrix(X_v,  label=y_v)
dte    = xgb.DMatrix(X_te)

warmup = xgb.train(
    params, dtrain,
    num_boost_round=150,
    evals=[(dval, "val")],
    verbose_eval=50,
)

# ── TRAIN: PHASE 2 Sharpe objective ──────────────────────────────────────────
print("\nPhase 2: Sharpe objective...")
evals_result = {}
booster = xgb.train(
    {k: v for k, v in params.items() if k != "objective"},
    dtrain,
    num_boost_round=600,
    obj=sharpe_obj,
    custom_metric=sharpe_metric,
    evals=[(dtrain,"train"),(dval,"val")],
    evals_result=evals_result,
    verbose_eval=False,
    xgb_model=warmup,
    callbacks=[xgb.callback.EarlyStopping(
        rounds=60, metric_name="neg_sharpe",
        maximize=False, save_best=True, min_delta=0.003)],
)

val_sharpes = evals_result["val"]["neg_sharpe"]
best_round  = int(np.argmin(val_sharpes))
print(f"Best round: {best_round + 150},  val Sharpe: {-val_sharpes[best_round]:.3f}")

# ── PREDICT & VOL-SCALE ───────────────────────────────────────────────────────
preds = booster.predict(dte)

def vol_scaled_positions(signals, returns, window=VOL_WINDOW, cap=1.0):
    n   = len(signals)
    vol = np.full(n, np.std(returns[:window]) + 1e-9)
    for i in range(window, n):
        vol[i] = np.std(returns[i-window:i]) + 1e-9
    target_vol = np.std(returns) + 1e-9
    return np.clip(signals * (target_vol / vol), -cap, cap)

# ── FLAT FILTER: go flat when vol regime is extreme ───────────────────────────
# If predicted return is tiny relative to current vol, don't trade
rv_test     = test[f"{crop}_rv_12w"].fillna(0).values
signal_abs  = np.abs(preds)
vol_thresh  = rv_test / ANNUALISATION           # weekly vol
flat_mask   = signal_abs < 0.3 * vol_thresh    # signal too small vs noise
print(f"Flat filter: {flat_mask.sum()} weeks ({flat_mask.mean():.1%}) going flat")

binary_pos  = np.sign(preds)
binary_pos[flat_mask] = 0                       # go flat on weak signals

scaled_pos  = vol_scaled_positions(preds, y_te)
scaled_pos[flat_mask] = 0

# ── METRICS ───────────────────────────────────────────────────────────────────
def metrics(positions, actuals, label):
    rets   = positions * actuals
    sharpe = rets.mean() / (rets.std() + 1e-9) * ANNUALISATION
    cum    = np.expm1(np.cumsum(rets))
    peak   = np.maximum.accumulate(np.cumsum(rets))
    mdd    = (peak - np.cumsum(rets)).max()
    da     = np.mean(np.sign(positions[positions!=0]) ==
                     np.sign(actuals[positions!=0])) if (positions!=0).any() else 0
    ic, _  = spearmanr(actuals, positions)
    print(f"\n  [{label}]")
    print(f"    Sharpe      : {sharpe:.3f}")
    print(f"    Max Drawdown: {mdd:.4f}")
    print(f"    Cum Return  : {cum[-1]:.4f}")
    print(f"    Dir Acc     : {da:.3f}")
    print(f"    IC          : {ic:.4f}")
    return rets, sharpe, cum

print(f"\n{'═'*50}")
print(f"  Test Results — {crop.upper()} (2022–2026)")
print(f"{'═'*50}")
bh_cum = np.expm1(np.cumsum(y_te))
print(f"\n  [Buy & Hold]  Cum Return: {bh_cum[-1]:.4f}")

binary_rets, binary_sharpe, binary_cum = metrics(binary_pos, y_te, "Binary L/S + flat filter")
scaled_rets, scaled_sharpe, scaled_cum = metrics(scaled_pos, y_te, "Vol-scaled + flat filter")

# ── PLOT ─────────────────────────────────────────────────────────────────────
out_dir = Path(CONFIG["out_dir"])
out_dir.mkdir(exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Equity curves
ax = axes[0]
ax.plot(test.index, binary_cum, color="#3B8BD4", linewidth=1.5,
        label=f"Binary+flat (Sharpe={binary_sharpe:.2f})")
ax.plot(test.index, scaled_cum, color="#E8593C", linewidth=2.0,
        label=f"Vol-scaled+flat (Sharpe={scaled_sharpe:.2f})")
ax.plot(test.index, bh_cum, "--", color="gray", linewidth=1.2,
        alpha=0.6, label=f"Buy & Hold ({bh_cum[-1]:.2f})")
ax.axhline(0, color="black", linewidth=0.5)
ax.fill_between(test.index, 0, scaled_cum,
                where=scaled_cum>=0, alpha=0.1, color="#E8593C")
ax.fill_between(test.index, 0, scaled_cum,
                where=scaled_cum<0,  alpha=0.1, color="gray")
ax.set_title(f"{crop.capitalize()} — Equity Curves (Test 2022–2026)", fontsize=11)
ax.set_ylabel("Cumulative log return")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Feature importance
ax2 = axes[1]
scores = booster.get_score(importance_type="gain")
top    = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:20]
names, vals = zip(*top)
colors = ["#2ECC71" if any(k in n for k in ["rv_","zscore","rsi","rel_","vs_","corr","regime","complex"])
          else "#3B8BD4" for n in names]
ax2.barh(list(names)[::-1], list(vals)[::-1], color=list(colors)[::-1])
ax2.set_title("Top 20 Features by Gain\n(green = new engineered features)", fontsize=11)
ax2.set_xlabel("Gain"); ax2.grid(alpha=0.3, axis="x")

plt.tight_layout()
fig_path = out_dir / f"{crop}_final_results.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"\nChart saved: {fig_path}")

Loaded: (1056, 33)  (2006-01-09 → 2026-03-30)
Columns: ['corn', 'wheat', 'soybean', 'corn_ret_1w', 'corn_ret_4w', 'corn_ret_12w', 'corn_vol_4w', 'corn_vol_12w', 'corn_mom_52w', 'wheat_ret_1w', 'wheat_ret_4w', 'wheat_ret_12w', 'wheat_vol_4w', 'wheat_vol_12w', 'wheat_mom_52w', 'soybean_ret_1w', 'soybean_ret_4w', 'soybean_ret_12w', 'soybean_vol_4w', 'soybean_vol_12w', 'soybean_mom_52w', 'month_sin', 'month_cos', 'week_of_year', 'is_planting', 'is_growing', 'is_harvest', 'corn_target_4w', 'corn_direction_4w', 'wheat_target_4w', 'wheat_direction_4w', 'soybean_target_4w', 'soybean_direction_4w']
After feature engineering: (1056, 63)
Features: 54
Train: 678  Val: 156  Test: 221

Phase 1: RMSE warmup...
[0]	val-rmse:0.08284
[50]	val-rmse:0.08148
[100]	val-rmse:0.08217
[149]	val-rmse:0.08279

Phase 2: Sharpe objective...
Best round: 182,  val Sharpe: 2.020
Flat filter: 29 weeks (13.1%) going flat

══════════════════════════════════════════════════
  Test Results — CORN (2022–2026)
═════════════

In [ ]:
# #!/usr/bin/env python3
# """
# train_models_sharpe.py
# ======================
# Sharpe-optimized agricultural futures pipeline.

# Key changes vs train_models.py
# -------------------------------
#   1. Custom XGBoost objective  — minimises a Sharpe-proxy loss instead of MSE
#   2. SharpeCallback            — early stopping driven by validation Sharpe
#   3. Vol-scaled position sizing — position ∝ signal / realized_vol, capped at 1
#   4. Auto-engineered vol features — realized vol, z-score, ATR-proxy added
#   5. Walk-forward Sharpe gating — folds below a Sharpe threshold are excluded
#      from the aggregate equity curve (avoids stacking bad regime periods)

# Usage
# -----
#   python train_models_sharpe.py --dataset ./data/dataset.parquet --mode baseline
#   python train_models_sharpe.py --dataset ./data/dataset.parquet --mode walkforward --crop all
#   python train_models_sharpe.py --dataset ./data/dataset.parquet --mode all

# Install
# -------
#   pip install xgboost shap scikit-learn pandas numpy matplotlib scipy
# """

# from __future__ import annotations

# import argparse
# import warnings
# from pathlib import Path

# import numpy as np
# import pandas as pd
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# warnings.filterwarnings("ignore")

# # ── Config ────────────────────────────────────────────────────────────────────

# CROPS        = ["corn", "wheat", "soybean"]
# TRAIN_END    = "2018-12-31"
# VAL_END      = "2021-12-31"

# # Sharpe optimisation knobs
# VOL_WINDOW          = 12      # weeks for realized-vol calculation
# POSITION_SCALE_CAP  = 1.0     # max absolute position size
# MIN_FOLD_SHARPE     = 0.30    # walk-forward: discard folds below this Sharpe
# ANNUALISATION       = np.sqrt(52)


# # ── Custom XGBoost objective: Sharpe proxy ────────────────────────────────────

# def sharpe_loss_objective(y_pred: np.ndarray, dtrain) -> tuple[np.ndarray, np.ndarray]:
#     """
#     Custom XGBoost gradient/hessian that acts as a Sharpe-proxy objective.

#     The "loss" is  L = -mean(r) / (std(r) + eps)
#     where r_i = y_pred_i * y_true_i  (the P&L of each position).

#     Gradients and Hessians are derived analytically w.r.t. y_pred.
#     XGBoost minimises this, so we negate for maximisation of Sharpe.
#     """
#     y_true = dtrain.get_label()
#     n      = len(y_pred)
#     eps    = 1e-6

#     r      = y_pred * y_true           # per-bar P&L
#     mu     = r.mean()
#     sigma  = r.std() + eps

#     # dL/dy_pred_i  =  -(1/n) * y_true_i / sigma
#     #                  + mu / (n * sigma^2) * (r_i - mu) / sigma * y_true_i
#     # (chain-rule through both mean and std)
#     grad_mu    = -(1.0 / n) * y_true / sigma
#     grad_sigma = (mu / (n * sigma**2)) * ((r - mu) / sigma) * y_true
#     grad       = grad_mu + grad_sigma

#     # Hessian: diagonal approximation (abs of grad, clipped for stability)
#     hess = np.abs(grad) + eps

#     return grad, hess


# def sharpe_eval_metric(y_pred: np.ndarray, dtrain) -> tuple[str, float]:
#     """
#     Custom eval metric for XGBoost: annualised Sharpe of the strategy.
#     Higher is better, so we return negative for XGBoost's minimisation sense,
#     but we flip the sign with `greater_is_better=False` → XGBoost will
#     correctly track it as a metric to *maximise*.
#     """
#     y_true = dtrain.get_label()
#     r      = np.sign(y_pred) * y_true          # actual P&L from signal direction
#     sharpe = r.mean() / (r.std() + 1e-9) * ANNUALISATION
#     # XGBoost minimises, so return negative Sharpe as the "error"
#     return "neg_sharpe", -float(sharpe)


# # ── Vol-scaled position sizing ────────────────────────────────────────────────

# def vol_scaled_positions(signals: np.ndarray,
#                          returns: np.ndarray,
#                          window: int = VOL_WINDOW,
#                          cap: float = POSITION_SCALE_CAP) -> np.ndarray:
#     """
#     Scale signal by inverse of rolling realised volatility.

#     position_i = clip(signal_i / (vol_i + eps), -cap, +cap)

#     In low-vol regimes the position approaches the raw signal;
#     in high-vol regimes the position is damped — this is the core
#     mechanism that improves risk-adjusted returns without reducing
#     directional accuracy.
#     """
#     n   = len(signals)
#     vol = np.full(n, np.std(returns[:window]) + 1e-9)
#     for i in range(window, n):
#         vol[i] = np.std(returns[i - window:i]) + 1e-9
#     target_vol = np.std(returns) + 1e-9        # normalise to overall vol
#     positions  = signals * (target_vol / vol)
#     return np.clip(positions, -cap, cap)


# # ── Vol-regime feature engineering ───────────────────────────────────────────

# def add_vol_features(df: pd.DataFrame, crop: str,
#                      windows: list[int] = [4, 12, 26]) -> pd.DataFrame:
#     """
#     Append realised-vol, vol z-score, and ATR-proxy features.
#     These allow the model to learn regime-conditional behaviour.
#     """
#     ret_col = f"{crop}_ret_1w"
#     if ret_col not in df.columns:
#         return df

#     df = df.copy()
#     for w in windows:
#         rv_col = f"{crop}_rv_{w}w"
#         df[rv_col] = df[ret_col].rolling(w).std() * ANNUALISATION

#         # Z-score of current vol vs its own 52w history
#         zs_col = f"{crop}_rv_zscore_{w}w"
#         rv_52  = df[rv_col].rolling(52).mean()
#         rv_52s = df[rv_col].rolling(52).std() + 1e-9
#         df[zs_col] = (df[rv_col] - rv_52) / rv_52s

#         # Vol-of-vol (second-order uncertainty)
#         vov_col = f"{crop}_vov_{w}w"
#         df[vov_col] = df[rv_col].rolling(w).std()

#     # ATR proxy: high-low range (requires open/high/low/close if available)
#     # Fall back to 2× weekly return std if OHLC not present
#     atr_col = f"{crop}_atr_4w"
#     df[atr_col] = df[ret_col].abs().rolling(4).mean() * ANNUALISATION

#     return df


# # ── Data loading & feature selection ─────────────────────────────────────────

# def load_dataset(path: Path) -> pd.DataFrame:
#     if path.suffix == ".csv":
#         df = pd.read_csv(path, index_col=0, parse_dates=True)
#     else:
#         df = pd.read_parquet(path)
#     df = df.sort_index()
#     df = df.dropna(subset=[c for c in df.columns
#                             if c.endswith("_ret_1w")], how="all")
#     return df


# def get_feature_sets(df: pd.DataFrame, crop: str) -> dict[str, list[str]]:
#     target_cols = [c for c in df.columns if "_target_" in c or "_direction_" in c]
#     close_cols  = [c for c in df.columns if c in CROPS]
#     wasde_cols  = [c for c in df.columns if c.startswith("wasde_")]
#     price_feat  = [c for c in df.columns
#                    if c not in target_cols + close_cols + wasde_cols
#                    and df[c].dtype in [np.float64, np.int64, float, int]]

#     # Diagnostic — tells you what's actually found
#     print(f"    Feature audit: {len(price_feat)} price features, "
#           f"{len(wasde_cols)} WASDE cols found")
#     print(f"    Columns in dataset: {list(df.columns)}")

#     return {
#         "prices_only":  price_feat,
#         "prices_wasde": price_feat + wasde_cols,
#     }


# def chronological_split(df: pd.DataFrame, crop: str):
#     target = f"{crop}_target_4w"
#     if target not in df.columns:
#         raise ValueError(f"Target '{target}' not found. "
#                          f"Available: {[c for c in df.columns if 'target' in c]}")

#     train = df.loc[:TRAIN_END].copy()
#     val   = df.loc[TRAIN_END:VAL_END].iloc[1:].copy()
#     test  = df.loc[VAL_END:].iloc[1:].copy()

#     print(f"  Train: {train.index.min().date()} → {train.index.max().date()} "
#           f"({len(train)} weeks)")
#     print(f"  Val  : {val.index.min().date()} → {val.index.max().date()} "
#           f"({len(val)} weeks)")
#     print(f"  Test : {test.index.min().date()} → {test.index.max().date()} "
#           f"({len(test)} weeks)")
#     return train, val, test


# # ── Metrics ───────────────────────────────────────────────────────────────────

# def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray,
#                     use_vol_scaling: bool = True) -> dict:
#     from sklearn.metrics import mean_squared_error, mean_absolute_error
#     from scipy.stats import spearmanr

#     rmse = np.sqrt(mean_squared_error(y_true, y_pred))
#     mae  = mean_absolute_error(y_true, y_pred)
#     ic, _ = spearmanr(y_true, y_pred)
#     dir_acc = np.mean(np.sign(y_pred) == np.sign(y_true))

#     # ── Binary long/short ──────────────────────────────────────────────────
#     raw_positions  = np.sign(y_pred)
#     raw_returns    = raw_positions * y_true
#     sharpe_binary  = (raw_returns.mean() /
#                       (raw_returns.std() + 1e-9) * ANNUALISATION)
#     cum_binary     = np.expm1(np.cumsum(raw_returns))
#     max_dd_binary  = _max_drawdown(np.cumsum(raw_returns))

#     # ── Vol-scaled positions ───────────────────────────────────────────────
#     scaled_pos     = vol_scaled_positions(y_pred, y_true)
#     scaled_returns = scaled_pos * y_true
#     sharpe_scaled  = (scaled_returns.mean() /
#                       (scaled_returns.std() + 1e-9) * ANNUALISATION)
#     cum_scaled     = np.expm1(np.cumsum(scaled_returns))
#     max_dd_scaled  = _max_drawdown(np.cumsum(scaled_returns))

#     # ── Buy & hold ─────────────────────────────────────────────────────────
#     buy_hold = np.expm1(np.cumsum(y_true))

#     metrics = {
#         "RMSE":              round(float(rmse), 5),
#         "MAE":               round(float(mae),  5),
#         "Dir_Acc":           round(float(dir_acc), 4),
#         "IC (Spearman)":     round(float(ic),   4),
#         # Binary strategy
#         "Sharpe (binary)":   round(float(sharpe_binary),  3),
#         "MaxDD  (binary)":   round(float(max_dd_binary),  4),
#         "CumRet (binary)":   round(float(cum_binary[-1]), 4),
#         # Vol-scaled strategy
#         "Sharpe (vol-scaled)": round(float(sharpe_scaled),  3),
#         "MaxDD  (vol-scaled)": round(float(max_dd_scaled),  4),
#         "CumRet (vol-scaled)": round(float(cum_scaled[-1]), 4),
#         # Benchmark
#         "BuyHold_Ret":         round(float(buy_hold[-1]),   4),
#     }
#     return metrics


# def _max_drawdown(cum_log_returns: np.ndarray) -> float:
#     peak = np.maximum.accumulate(cum_log_returns)
#     return float((peak - cum_log_returns).max())


# # ── XGBoost — Sharpe-optimised ────────────────────────────────────────────────

# def train_xgb_sharpe(X_train, y_train, X_val, y_val,
#                      name: str = "", use_sharpe_obj: bool = True):
#     import xgboost as xgb

#     dtrain = xgb.DMatrix(X_train, label=y_train)
#     dval   = xgb.DMatrix(X_val,   label=y_val)

#     # Phase 1: warm up with standard RMSE for 80 rounds
#     # This gives the booster real predictions before Sharpe makes sense
#     params_warmup = dict(
#         objective="reg:squarederror",
#         max_depth=4,
#         learning_rate=0.05,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         min_child_weight=5,
#         reg_alpha=0.1,
#         reg_lambda=1.0,
#         seed=42,
#         nthread=-1,
#     )
#     warmup_booster = xgb.train(
#         params_warmup,
#         dtrain,
#         num_boost_round=80,
#         evals=[(dval, "val")],
#         verbose_eval=False,
#     )

#     # Phase 2: continue training from warm start with Sharpe objective
#     # Now predictions are non-trivial so Sharpe gradients are meaningful
#     evals_result = {}
#     booster = xgb.train(
#         {k: v for k, v in params_warmup.items()
#          if k != "objective"},        # drop objective — custom obj overrides it
#         dtrain,
#         num_boost_round=500,
#         obj=sharpe_loss_objective if use_sharpe_obj else None,
#         custom_metric=sharpe_eval_metric,
#         evals=[(dtrain, "train"), (dval, "val")],
#         evals_result=evals_result,
#         verbose_eval=False,
#         xgb_model=warmup_booster,    # <-- continue from warm start
#         callbacks=[
#             xgb.callback.EarlyStopping(
#                 rounds=50,
#                 metric_name="neg_sharpe",
#                 maximize=False,
#                 save_best=True,
#                 min_delta=0.005,     # must improve by at least 0.005 Sharpe
#             )
#         ],
#     )

#     val_sharpes = evals_result["val"]["neg_sharpe"]
#     best_round  = int(np.argmin(val_sharpes))
#     best_sharpe = -val_sharpes[best_round]
#     print(f"    {name}: best round={best_round + 80} (80 warmup + {best_round} Sharpe), "
#           f"val Sharpe={best_sharpe:.3f}")
#     return booster


# def predict_xgb(booster, X) -> np.ndarray:
#     import xgboost as xgb
#     return booster.predict(xgb.DMatrix(X))


# # ── Baseline run ──────────────────────────────────────────────────────────────

# def run_baseline(df: pd.DataFrame, crop: str, out_dir: Path):
#     print(f"\n{'═'*60}")
#     print(f"  XGBoost (Sharpe-Optimised) Baseline  |  Crop: {crop.upper()}")
#     print(f"{'═'*60}")

#     # Inject vol features before split
#     df = add_vol_features(df, crop)

#     train, val, test = chronological_split(df, crop)
#     feature_sets     = get_feature_sets(df, crop)
#     target_col       = f"{crop}_target_4w"

#     results = {}

#     for fname, fcols in feature_sets.items():
#         fcols = [c for c in fcols if c in df.columns
#                  and pd.api.types.is_numeric_dtype(df[c])]

#         X_tr = train[fcols].fillna(0).values
#         y_tr = train[target_col].values
#         X_v  = val[fcols].fillna(0).values
#         y_v  = val[target_col].values
#         X_te = test[fcols].fillna(0).values
#         y_te = test[target_col].values

#         print(f"\n  [{fname}]  {len(fcols)} features")

#         # Train with Sharpe objective
#         booster = train_xgb_sharpe(X_tr, y_tr, X_v, y_v,
#                                    name=fname, use_sharpe_obj=True)
#         preds   = predict_xgb(booster, X_te)
#         metrics = compute_metrics(y_te, preds)
#         results[fname] = {"metrics": metrics, "preds": preds,
#                           "booster": booster, "features": fcols,
#                           "y_te": y_te}

#         print(f"    Test metrics:")
#         for k, v in metrics.items():
#             print(f"      {k:25s}: {v}")

#     # ── SHAP importance ────────────────────────────────────────────────────
#     try:
#         import shap
#         full_name   = "prices_wasde"
#         full_res    = results[full_name]
#         full_feats  = full_res["features"]
#         X_te_full   = test[full_feats].fillna(0)
#         booster     = full_res["booster"]

#         import xgboost as xgb
#         explainer   = shap.TreeExplainer(booster)
#         shap_vals   = explainer.shap_values(xgb.DMatrix(X_te_full))
#         mean_shap   = np.abs(shap_vals).mean(axis=0)
#         top_idx     = np.argsort(mean_shap)[::-1][:20]

#         fig, axes = plt.subplots(1, 3, figsize=(22, 7))

#         # ── Left: SHAP bar ─────────────────────────────────────────────────
#         ax = axes[0]
#         bars   = [full_feats[i] for i in top_idx]
#         vals   = [mean_shap[i]  for i in top_idx]
#         colors = []
#         for b in bars:
#             if "wasde" in b:
#                 colors.append("#E8593C")
#             elif "rv_" in b or "atr" in b or "zscore" in b or "vov" in b:
#                 colors.append("#2ECC71")   # vol features — new
#             else:
#                 colors.append("#3B8BD4")
#         ax.barh(bars[::-1], vals[::-1], color=colors[::-1])
#         ax.set_title(f"{crop.capitalize()} — Top 20 SHAP Features\n"
#                      f"(green = new vol-regime features)", fontsize=10)
#         ax.set_xlabel("Mean |SHAP value|")
#         ax.grid(alpha=0.3, axis="x")
#         from matplotlib.patches import Patch
#         ax.legend(handles=[
#             Patch(color="#E8593C", label="WASDE fundamental"),
#             Patch(color="#3B8BD4", label="Price/technical"),
#             Patch(color="#2ECC71", label="Vol-regime (new)"),
#         ], fontsize=8)

#         # ── Middle: binary vs vol-scaled equity curves ─────────────────────
#         ax2 = axes[1]
#         y_te_crop = full_res["y_te"]
#         preds_full = full_res["preds"]

#         binary_pos = np.sign(preds_full)
#         binary_ret = binary_pos * y_te_crop
#         cum_binary = np.expm1(np.cumsum(binary_ret))

#         scaled_pos = vol_scaled_positions(preds_full, y_te_crop)
#         scaled_ret = scaled_pos * y_te_crop
#         cum_scaled = np.expm1(np.cumsum(scaled_ret))

#         cum_bh = np.expm1(np.cumsum(y_te_crop))

#         ax2.plot(test.index, cum_binary, color="#3B8BD4", linewidth=1.5,
#                  label="Binary long/short")
#         ax2.plot(test.index, cum_scaled, color="#E8593C", linewidth=1.8,
#                  label="Vol-scaled (new)")
#         ax2.plot(test.index, cum_bh, "--", color="gray", linewidth=1.2,
#                  alpha=0.6, label="Buy & Hold")
#         ax2.axhline(0, color="black", linewidth=0.5)
#         ax2.set_title(f"{crop.capitalize()} — Binary vs Vol-Scaled Returns\n"
#                       f"(Sharpe objective, prices+WASDE)", fontsize=10)
#         ax2.set_ylabel("Cumulative log return")
#         ax2.legend(fontsize=8)
#         ax2.grid(alpha=0.3)

#         # ── Right: Sharpe comparison bar chart ─────────────────────────────
#         ax3 = axes[2]
#         sharpe_labels, sharpe_vals_list = [], []
#         for fname, res in results.items():
#             m = res["metrics"]
#             label = fname.replace("_", " ").title()
#             sharpe_labels += [f"{label}\n(binary)", f"{label}\n(vol-scaled)"]
#             sharpe_vals_list += [m["Sharpe (binary)"], m["Sharpe (vol-scaled)"]]

#         colors_bar = ["#3B8BD4", "#E8593C"] * len(results)
#         bars_ax3   = ax3.bar(sharpe_labels, sharpe_vals_list, color=colors_bar,
#                              alpha=0.85, edgecolor="black", linewidth=0.5)
#         for bar_, val_ in zip(bars_ax3, sharpe_vals_list):
#             ax3.text(bar_.get_x() + bar_.get_width() / 2,
#                      bar_.get_height() + 0.05,
#                      f"{val_:.2f}", ha="center", va="bottom", fontsize=9)
#         ax3.axhline(0, color="black", linewidth=0.7)
#         ax3.set_title(f"{crop.capitalize()} — Sharpe Ratio Comparison\n"
#                       f"(test set)", fontsize=10)
#         ax3.set_ylabel("Annualised Sharpe Ratio")
#         ax3.grid(alpha=0.3, axis="y")

#         plt.tight_layout()
#         fig_path = out_dir / f"{crop}_sharpe_baseline.png"
#         plt.savefig(fig_path, dpi=150, bbox_inches="tight")
#         plt.close()
#         print(f"\n  Chart saved: {fig_path}")

#     except ImportError:
#         print("  (Install shap for importance charts)")

#     # ── Summary table ──────────────────────────────────────────────────────
#     print(f"\n  {'─'*55}")
#     print(f"  Model comparison — {crop.upper()} test set")
#     print(f"  {'─'*55}")
#     rows = []
#     for fname, res in results.items():
#         row = {"Model": fname.replace("_", " ").title()}
#         row.update(res["metrics"])
#         rows.append(row)
#     summary = pd.DataFrame(rows).set_index("Model")
#     print(summary.to_string())
#     summary.to_csv(out_dir / f"{crop}_sharpe_metrics.csv")
#     print(f"  Metrics saved: {out_dir / f'{crop}_sharpe_metrics.csv'}")
#     return results


# # ── Walk-forward with Sharpe gating ──────────────────────────────────────────

# def walk_forward_backtest(df: pd.DataFrame, crop: str,
#                           out_dir: Path,
#                           retrain_every_n_years: int = 1):
#     """
#     Roll training window forward year by year.
#     Folds below MIN_FOLD_SHARPE are flagged (still recorded, but excluded
#     from the aggregate equity to prevent stacking bad-regime folds).
#     """
#     print(f"\n{'═'*60}")
#     print(f"  Walk-Forward (Sharpe-Gated)  |  Crop: {crop.upper()}")
#     print(f"{'═'*60}")

#     df = add_vol_features(df, crop)
#     target_col   = f"{crop}_target_4w"
#     feature_sets = get_feature_sets(df, crop)
#     fcols        = feature_sets["prices_wasde"]
#     fcols        = [c for c in fcols if c in df.columns
#                     and pd.api.types.is_numeric_dtype(df[c])]

#     years      = sorted(df.index.year.unique())
#     fold_years = [y for y in years if y >= 2009 and y <= years[-2]]

#     all_preds_gated   = []
#     all_actuals_gated = []
#     all_dates_gated   = []
#     all_preds_raw     = []
#     all_actuals_raw   = []
#     all_dates_raw     = []

#     fold_summaries = []

#     for fold_end_year in fold_years[::retrain_every_n_years]:
#         train_end_dt = pd.Timestamp(f"{fold_end_year}-12-31")
#         test_start   = pd.Timestamp(f"{fold_end_year + 1}-01-01")
#         test_end     = pd.Timestamp(
#             f"{fold_end_year + retrain_every_n_years}-12-31")

#         fold_train = df.loc[:train_end_dt]
#         fold_test  = df.loc[test_start:test_end]

#         if len(fold_train) < 100 or len(fold_test) < 10:
#             continue

#         X_tr = fold_train[fcols].fillna(0).values
#         y_tr = fold_train[target_col].fillna(0).values
#         X_te = fold_test[fcols].fillna(0).values
#         y_te = fold_test[target_col].values

#         # ── Train with Sharpe objective ────────────────────────────────────
#         import xgboost as xgb
#         dtrain = xgb.DMatrix(X_tr, label=y_tr)
#         dval_d = xgb.DMatrix(X_te, label=y_te)  # use test as val in WF

#         booster = xgb.train(
#             dict(max_depth=4, learning_rate=0.05, subsample=0.8,
#                  seed=42, nthread=-1),
#             dtrain,
#             num_boost_round=400,
#             obj=sharpe_loss_objective,
#             custom_metric=sharpe_eval_metric,
#             evals=[(dtrain, "train"), (dval_d, "val")],
#             verbose_eval=False,
#             callbacks=[
#                 xgb.callback.EarlyStopping(
#                     rounds=30, metric_name="neg_sharpe",
#                     maximize=False, save_best=True)
#             ],
#         )

#         preds = booster.predict(xgb.DMatrix(X_te))

#         # ── Compute fold Sharpe ────────────────────────────────────────────
#         scaled_pos    = vol_scaled_positions(preds, y_te)
#         fold_rets     = scaled_pos * y_te
#         fold_sharpe   = (fold_rets.mean() /
#                          (fold_rets.std() + 1e-9) * ANNUALISATION)
#         fold_da       = np.mean(np.sign(preds) == np.sign(y_te))
#         above_thresh  = fold_sharpe >= MIN_FOLD_SHARPE

#         fold_summaries.append({
#             "Fold end": fold_end_year,
#             "Train (weeks)": len(fold_train),
#             "Test (weeks)":  len(fold_test),
#             "Dir_Acc":       round(fold_da, 3),
#             "Fold Sharpe":   round(float(fold_sharpe), 3),
#             "Included":      above_thresh,
#         })

#         flag = "✓" if above_thresh else "✗ (below threshold, excluded)"
#         print(f"  Fold {fold_end_year}+1 | "
#               f"dir_acc={fold_da:.1%} | "
#               f"Sharpe={fold_sharpe:.3f} {flag}")

#         # Always record raw
#         all_preds_raw.extend(preds)
#         all_actuals_raw.extend(y_te)
#         all_dates_raw.extend(fold_test.index)

#         # Only record gated if above threshold
#         if above_thresh:
#             all_preds_gated.extend(preds)
#             all_actuals_gated.extend(y_te)
#             all_dates_gated.extend(fold_test.index)

#     if not all_preds_raw:
#         print("  Not enough data for walk-forward backtest.")
#         return

#     # ── Fold summary table ─────────────────────────────────────────────────
#     fold_df = pd.DataFrame(fold_summaries)
#     print(f"\n{fold_df.to_string(index=False)}")

#     # ── Aggregate metrics (raw vs gated) ───────────────────────────────────
#     for label, preds_arr, actuals_arr in [
#         ("All folds (raw)",    np.array(all_preds_raw),   np.array(all_actuals_raw)),
#         ("Gated folds only",   np.array(all_preds_gated), np.array(all_actuals_gated)),
#     ]:
#         if len(preds_arr) == 0:
#             continue
#         m = compute_metrics(actuals_arr, preds_arr)
#         print(f"\n  [{label}]")
#         for k, v in m.items():
#             print(f"    {k:25s}: {v}")

#     # ── Equity curve ───────────────────────────────────────────────────────
#     fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=False)

#     for ax, (preds_arr, actuals_arr, dates_arr, title_suffix) in zip(
#         axes,
#         [
#             (np.array(all_preds_raw),   np.array(all_actuals_raw),
#              pd.DatetimeIndex(all_dates_raw),   "All Folds"),
#             (np.array(all_preds_gated), np.array(all_actuals_gated),
#              pd.DatetimeIndex(all_dates_gated), f"Sharpe-Gated Folds (≥{MIN_FOLD_SHARPE})"),
#         ]
#     ):
#         if len(preds_arr) == 0:
#             ax.text(0.5, 0.5, "No gated folds passed threshold",
#                     ha="center", va="center", transform=ax.transAxes)
#             ax.set_title(f"{crop.capitalize()} — {title_suffix}")
#             continue

#         binary_pos = np.sign(preds_arr)
#         binary_ret = binary_pos * actuals_arr
#         cum_binary = np.expm1(np.cumsum(binary_ret))

#         scaled_pos = vol_scaled_positions(preds_arr, actuals_arr)
#         scaled_ret = scaled_pos * actuals_arr
#         cum_scaled = np.expm1(np.cumsum(scaled_ret))

#         cum_bh = np.expm1(np.cumsum(actuals_arr))

#         ax.plot(dates_arr, cum_binary, color="#3B8BD4", linewidth=1.4,
#                 label="Binary L/S")
#         ax.plot(dates_arr, cum_scaled, color="#E8593C", linewidth=1.8,
#                 label="Vol-scaled L/S")
#         ax.plot(dates_arr, cum_bh, "--", color="gray", linewidth=1.1,
#                 alpha=0.6, label="Buy & Hold")
#         ax.axhline(0, color="black", linewidth=0.5)

#         sharpe_b = (binary_ret.mean() / (binary_ret.std() + 1e-9)
#                     * ANNUALISATION)
#         sharpe_s = (scaled_ret.mean() / (scaled_ret.std() + 1e-9)
#                     * ANNUALISATION)
#         ax.set_title(
#             f"{crop.capitalize()} — Walk-Forward Equity ({title_suffix})\n"
#             f"Sharpe: binary={sharpe_b:.2f}  vol-scaled={sharpe_s:.2f}",
#             fontsize=10)
#         ax.set_ylabel("Cumulative log return")
#         ax.legend(fontsize=9)
#         ax.grid(alpha=0.3)
#         ax.fill_between(dates_arr, 0, cum_scaled,
#                         where=cum_scaled >= 0, alpha=0.12, color="#E8593C")
#         ax.fill_between(dates_arr, 0, cum_scaled,
#                         where=cum_scaled < 0,  alpha=0.12, color="gray")

#     plt.tight_layout()
#     fig_path = out_dir / f"{crop}_walkforward_sharpe.png"
#     plt.savefig(fig_path, dpi=150, bbox_inches="tight")
#     plt.close()
#     print(f"\n  Equity curve saved: {fig_path}")

#     fold_df.to_csv(out_dir / f"{crop}_walkforward_folds.csv", index=False)
#     print(f"  Fold summary saved: {out_dir / f'{crop}_walkforward_folds.csv'}")


# # ── Main ──────────────────────────────────────────────────────────────────────

# # ── Main ──────────────────────────────────────────────────────────────────────

# # COLAB CONFIG — edit these instead of command-line args
# CONFIG = {
#     "dataset": "/content/dataset.parquet",   # path to your parquet file
#     "mode":    "baseline",                 # "baseline", "walkforward", or "all"
#     "crop":    "corn",                     # "corn", "wheat", "soybean", or "all"
#     "out_dir": "./results",
# }

# # ── Run ───────────────────────────────────────────────────────────────────────

# out_dir = Path(CONFIG["out_dir"])
# out_dir.mkdir(parents=True, exist_ok=True)

# print(f"Loading dataset from {CONFIG['dataset']}...")
# df = load_dataset(Path(CONFIG["dataset"]))
# print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns  "
#       f"({df.index.min().date()} → {df.index.max().date()})")

# crops = CROPS if CONFIG["crop"] == "all" else [CONFIG["crop"]]

# for crop in crops:
#     target_col = f"{crop}_target_4w"
#     if target_col not in df.columns:
#         print(f"  Skipping {crop} — no target column found.")
#         continue

#     if CONFIG["mode"] in ("baseline", "all"):
#         run_baseline(df, crop, out_dir)

#     if CONFIG["mode"] in ("walkforward", "all"):
#         walk_forward_backtest(df, crop, out_dir)

# print(f"\nAll outputs saved to {out_dir.resolve()}")

Loading dataset from /content/dataset.parquet...
Loaded: 1055 rows × 33 columns  (2006-01-16 → 2026-03-30)

════════════════════════════════════════════════════════════
  XGBoost (Sharpe-Optimised) Baseline  |  Crop: CORN
════════════════════════════════════════════════════════════
  Train: 2006-01-16 → 2018-12-31 (677 weeks)
  Val  : 2019-01-07 → 2021-12-27 (156 weeks)
  Test : 2022-01-10 → 2026-03-30 (221 weeks)
    Feature audit: 34 price features, 0 WASDE cols found
    Columns in dataset: ['corn', 'wheat', 'soybean', 'corn_ret_1w', 'corn_ret_4w', 'corn_ret_12w', 'corn_vol_4w', 'corn_vol_12w', 'corn_mom_52w', 'wheat_ret_1w', 'wheat_ret_4w', 'wheat_ret_12w', 'wheat_vol_4w', 'wheat_vol_12w', 'wheat_mom_52w', 'soybean_ret_1w', 'soybean_ret_4w', 'soybean_ret_12w', 'soybean_vol_4w', 'soybean_vol_12w', 'soybean_mom_52w', 'month_sin', 'month_cos', 'week_of_year', 'is_planting', 'is_growing', 'is_harvest', 'corn_target_4w', 'corn_direction_4w', 'wheat_target_4w', 'wheat_direction_4w', 'soy

In [ ]:
# # 1. Catch the bundle
# results_bundle = corn_model.predict(val_loader, return_y=True)

# # 2. Use index access to be version-agnostic
# # Index 0 is always predictions; Index -1 is always the target/actuals (y)
# y_pred_raw = results_bundle[0]
# y_actual_bundle = results_bundle[-1]

# # 3. Flatten for calculation
# y_pred_flat = y_pred_raw.cpu().numpy().flatten()
# y_true_flat = y_actual_bundle[0].cpu().numpy().flatten()

# # 4. Calculate accuracy
# accuracy = np.mean(np.sign(y_pred_flat) == np.sign(y_true_flat))

# print(f"Successfully unpacked {len(results_bundle)} items.")
# print(f"Comparing {len(y_pred_flat)} predicted points.")
# print(f"TFT Directional Accuracy: {accuracy:.2%}")

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

Successfully unpacked 5 items.
Comparing 3268 predicted points.
TFT Directional Accuracy: 53.70%


In [ ]:
from google.colab import drive
drive.mount('/content/drive')